# Power BI Workspace & Dataset ID Discovery Tool

**Purpose:** Find workspace IDs and dataset IDs for Power BI semantic models to use in pipeline refresh notebooks.

**Use Case:** When adding new reports to the pipeline, you need the workspace_id and dataset_id parameters.

**Authentication:** Uses the same Power BI token method as the Universal Semantic Model Refresh notebook.

---

## How to Use

1. **Update Parameters** in Cell 2:
   - Set `WORKSPACE_NAME` to your target workspace (e.g., "RP - Parts Reports")
   - Set `DATASET_NAME` to your semantic model name (e.g., "Combine Vault Sales")

2. **Run All Cells** (Shift + Enter through each, or Run All)

3. **Copy the Output** - The final cell outputs the IDs in a format ready for pipeline parameters

---

In [ ]:
# Cell 1: Import Required Libraries
from notebookutils import mssparkutils
import requests
import json
import pandas as pd

print("✅ Libraries imported successfully")

In [ ]:
# Cell 2: PARAMETERS - Update These!

# Target workspace name (exact match, case-sensitive)
WORKSPACE_NAME = "RP - Parts Reports"

# Target dataset/semantic model name (exact match, case-sensitive)
DATASET_NAME = "Combine Vault Sales"

# Optional: Set to True to list ALL workspaces and datasets (useful for discovery)
LIST_ALL = False

print("🎯 Parameters Set:")
print(f"   Searching for Workspace: '{WORKSPACE_NAME}'")
print(f"   Searching for Dataset: '{DATASET_NAME}'")
print(f"   List All Mode: {LIST_ALL}")

In [ ]:
# Cell 3: Authenticate to Power BI

print("🔐 Getting Power BI authentication token...")

try:
    token = mssparkutils.credentials.getToken("pbi")
    
    headers = {
        "Authorization": f"Bearer {token}",
        "Content-Type": "application/json"
    }
    
    print("✅ Authentication successful")
    print(f"   Token length: {len(token)} characters")
    
except Exception as e:
    print(f"❌ Authentication failed: {str(e)}")
    raise

In [ ]:
# Cell 4: List All Workspaces (Groups)

print("\n" + "="*80)
print("📁 DISCOVERING WORKSPACES")
print("="*80)

# API endpoint for listing workspaces
# Note: Power BI calls workspaces "groups" in the API
workspaces_url = "https://api.powerbi.com/v1.0/myorg/groups"

try:
    response = requests.get(workspaces_url, headers=headers)
    
    if response.status_code == 200:
        workspaces_data = response.json()
        workspaces = workspaces_data.get('value', [])
        
        print(f"✅ Found {len(workspaces)} workspaces you have access to\n")
        
        # Create DataFrame for easy viewing
        workspace_df = pd.DataFrame([
            {
                'Workspace Name': ws.get('name'),
                'Workspace ID': ws.get('id'),
                'Type': ws.get('type', 'Workspace'),
                'State': ws.get('state', 'Active')
            }
            for ws in workspaces
        ])
        
        if LIST_ALL:
            print("📋 All Workspaces:")
            print(workspace_df.to_string(index=False))
            print()
        
        # Search for target workspace
        target_workspace = next((ws for ws in workspaces if ws.get('name') == WORKSPACE_NAME), None)
        
        if target_workspace:
            workspace_id = target_workspace.get('id')
            print(f"🎯 Target Workspace Found:")
            print(f"   Name: {target_workspace.get('name')}")
            print(f"   ID: {workspace_id}")
            print(f"   Type: {target_workspace.get('type', 'Workspace')}")
            print(f"   State: {target_workspace.get('state', 'Active')}")
        else:
            print(f"⚠️  Workspace '{WORKSPACE_NAME}' not found!")
            print(f"\n💡 Tip: Available workspaces containing 'Parts':")
            parts_workspaces = workspace_df[workspace_df['Workspace Name'].str.contains('Parts', case=False, na=False)]
            if not parts_workspaces.empty:
                print(parts_workspaces.to_string(index=False))
            else:
                print("   (None found - check spelling and access permissions)")
            workspace_id = None
            
    else:
        print(f"❌ Failed to retrieve workspaces: Status {response.status_code}")
        print(f"   Response: {response.text}")
        workspace_id = None
        
except Exception as e:
    print(f"❌ Error retrieving workspaces: {str(e)}")
    workspace_id = None
    raise

In [ ]:
# Cell 5: List Datasets in Target Workspace

if workspace_id:
    print("\n" + "="*80)
    print("📊 DISCOVERING DATASETS/SEMANTIC MODELS")
    print("="*80)
    
    # API endpoint for listing datasets in a workspace
    datasets_url = f"https://api.powerbi.com/v1.0/myorg/groups/{workspace_id}/datasets"
    
    try:
        response = requests.get(datasets_url, headers=headers)
        
        if response.status_code == 200:
            datasets_data = response.json()
            datasets = datasets_data.get('value', [])
            
            print(f"✅ Found {len(datasets)} datasets in '{WORKSPACE_NAME}'\n")
            
            # Create DataFrame for easy viewing
            dataset_df = pd.DataFrame([
                {
                    'Dataset Name': ds.get('name'),
                    'Dataset ID': ds.get('id'),
                    'Configured By': ds.get('configuredBy', 'N/A'),
                    'Is Refreshable': ds.get('isRefreshable', False)
                }
                for ds in datasets
            ])
            
            if LIST_ALL:
                print("📋 All Datasets in Workspace:")
                print(dataset_df.to_string(index=False))
                print()
            
            # Search for target dataset
            target_dataset = next((ds for ds in datasets if ds.get('name') == DATASET_NAME), None)
            
            if target_dataset:
                dataset_id = target_dataset.get('id')
                print(f"🎯 Target Dataset Found:")
                print(f"   Name: {target_dataset.get('name')}")
                print(f"   ID: {dataset_id}")
                print(f"   Configured By: {target_dataset.get('configuredBy', 'N/A')}")
                print(f"   Is Refreshable: {target_dataset.get('isRefreshable', False)}")
                print(f"   Create Report Embed URL: {target_dataset.get('createReportEmbedURL', 'N/A')}")
            else:
                print(f"⚠️  Dataset '{DATASET_NAME}' not found in this workspace!")
                print(f"\n💡 Tip: Available datasets:")
                if not dataset_df.empty:
                    print(dataset_df[['Dataset Name', 'Dataset ID']].to_string(index=False))
                else:
                    print("   (No datasets found in this workspace)")
                dataset_id = None
                
        else:
            print(f"❌ Failed to retrieve datasets: Status {response.status_code}")
            print(f"   Response: {response.text}")
            dataset_id = None
            
    except Exception as e:
        print(f"❌ Error retrieving datasets: {str(e)}")
        dataset_id = None
        raise
else:
    print("\n⚠️  Skipping dataset discovery - workspace not found")
    dataset_id = None

In [ ]:
# Cell 6: Output Summary - Copy These Values!

print("\n" + "="*80)
print("📋 RESULTS SUMMARY")
print("="*80)

if workspace_id and dataset_id:
    print("\n✅ SUCCESS - All IDs Found!\n")
    
    print("─"*80)
    print("📁 WORKSPACE DETAILS")
    print("─"*80)
    print(f"Workspace Name: {WORKSPACE_NAME}")
    print(f"Workspace ID:   {workspace_id}")
    print()
    
    print("─"*80)
    print("📊 DATASET DETAILS")
    print("─"*80)
    print(f"Dataset Name: {DATASET_NAME}")
    print(f"Dataset ID:   {dataset_id}")
    print()
    
    print("─"*80)
    print("📝 PIPELINE PARAMETERS - Copy These Values!")
    print("─"*80)
    print("\nFor Notebook Activity Parameters in your pipeline:\n")
    print("{")
    print(f'  "workspace_id": "{workspace_id}",')  
    print(f'  "dataset_id": "{dataset_id}",')  
    print(f'  "semantic_model_name": "{DATASET_NAME}"')  
    print("}")
    print()
    
    print("─"*80)
    print("🔗 QUICK LINKS")
    print("─"*80)
    print(f"Workspace URL:")
    print(f"  https://app.powerbi.com/groups/{workspace_id}/list")
    print()
    print(f"Dataset Settings URL:")
    print(f"  https://app.powerbi.com/groups/{workspace_id}/datasets/{dataset_id}/details")
    print()
    
    print("─"*80)
    print("✅ NEXT STEPS")
    print("─"*80)
    print("1. Copy the parameters JSON above")
    print("2. Add a new Notebook activity to your Pipeline_Facts_Inspections")
    print("3. Configure it to use 'Universal_SemanticModel_Refresh' notebook")
    print("4. Paste these parameters into the Notebook activity")
    print("5. Set dependencies: Wait for all fact dataflows to complete")
    print("6. Test the pipeline!")
    print()
    
elif workspace_id and not dataset_id:
    print("\n⚠️  PARTIAL SUCCESS - Workspace found, but dataset not found")
    print(f"\nWorkspace ID: {workspace_id}")
    print(f"\n💡 Check:")
    print(f"   1. Dataset name spelling: '{DATASET_NAME}'")
    print(f"   2. Dataset exists in workspace: '{WORKSPACE_NAME}'")
    print(f"   3. You have access to the dataset")
    print(f"\n   Review the 'All Datasets in Workspace' list above for available options.")
    
elif not workspace_id:
    print("\n❌ FAILED - Workspace not found")
    print(f"\n💡 Check:")
    print(f"   1. Workspace name spelling: '{WORKSPACE_NAME}'")
    print(f"   2. You have access to the workspace")
    print(f"   3. Review the workspaces list above")
    print(f"\n   Try setting LIST_ALL = True in Cell 2 to see all available workspaces.")

print("\n" + "="*80)
print("🏁 Discovery Complete")
print("="*80)